# Medical Device Risk Class Prediction — Model Training
### Cognizant AI Hackathon — Team Notebook

**Input**: `outputs/train.csv`, `outputs/test.csv`, `outputs/preprocessor.pkl` (from `preprocessing.ipynb`)

**Output**: `backend/app/ml/model.pkl`, `outputs/metrics.json`

**Evaluation metric**: Macro-F1 (not accuracy — class imbalance: II≈76%, I≈17%, III≈7%)

**Model progression**:
1. Logistic Regression — interpretable baseline, unblocks Day-1 API demo
2. Random Forest — non-linear baseline
3. XGBoost — final model for presentation


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style('whitegrid')

OUT_DIR        = Path('outputs')
BACKEND_ML_DIR = Path('../backend/app/ml')
BACKEND_ML_DIR.mkdir(parents=True, exist_ok=True)

# ── Load preprocessor (fit only on train — never refit here) ──────────────────
preprocessor = joblib.load(OUT_DIR / 'preprocessor.pkl')

# ── Load splits ───────────────────────────────────────────────────────────────
train_df = pd.read_csv(OUT_DIR / 'train.csv')
test_df  = pd.read_csv(OUT_DIR / 'test.csv')

FEATURE_COLS = [
    'classification', 'description',
    'mfr_total_events', 'mfr_distinct_countries',
    'mfr_distinct_devices_recalled', 'mfr_pct_class1_events',
    'description_len',
]

X_train = train_df[FEATURE_COLS]
y_train = train_df['risk_class']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['risk_class']

# ── Transform (preprocessor already fitted on train in preprocessing.ipynb) ───
X_train_t = preprocessor.transform(X_train)
X_test_t  = preprocessor.transform(X_test)

print(f'Train: {X_train_t.shape}  |  Test: {X_test_t.shape}')
print(f'Classes: {sorted(y_train.unique())}')

## Helper — evaluate any trained model

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """Print classification report + confusion matrix; return metrics dict."""
    y_pred = model.predict(X_te)

    macro_f1   = f1_score(y_te, y_pred, average='macro')
    report     = classification_report(y_te, y_pred, output_dict=True)
    report_str = classification_report(y_te, y_pred)

    print(f'\n{'='*60}')
    print(f'  {name}')
    print(f'{'='*60}')
    print(report_str)

    # Confusion matrix
    cm = confusion_matrix(y_te, y_pred, labels=[1, 2, 3])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=['I', 'II', 'III']).plot(ax=ax, colorbar=False)
    ax.set_title(f'{name} — Confusion Matrix (test set)')
    plt.tight_layout()
    safe_name = name.lower().replace(' ', '_')
    plt.savefig(OUT_DIR / f'cm_{safe_name}.png', dpi=110)
    plt.show()

    return {
        'model_name': name,
        'macro_f1':   round(macro_f1, 4),
        'per_class':  {
            str(cls): {
                'precision': round(report[str(cls)]['precision'], 4),
                'recall':    round(report[str(cls)]['recall'],    4),
                'f1':        round(report[str(cls)]['f1-score'],  4),
            }
            for cls in [1, 2, 3]
        },
    }

all_metrics = []

## 1. Logistic Regression — interpretable baseline

Fast to train, gives a working end-to-end demo on Day 1 while heavier models are still tuning.

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # compensates for II≈76% dominance
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train_t, y_train)

lr_metrics = evaluate('Logistic Regression', lr, X_train_t, y_train, X_test_t, y_test)
all_metrics.append(lr_metrics)

## 2. Random Forest — non-linear baseline

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_t, y_train)

rf_metrics = evaluate('Random Forest', rf, X_train_t, y_train, X_test_t, y_test)
all_metrics.append(rf_metrics)

## 3. XGBoost — final model

XGBoost handles sparse TF-IDF matrices well and typically outperforms RF on tabular + text hybrid features.

`scale_pos_weight` is not used here because we have 3 classes; instead we pass `sample_weight` derived from sklearn's `compute_sample_weight`.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight('balanced', y_train)

xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)
# XGBoost expects labels 0-indexed; map 1→0, 2→1, 3→2
xgb.fit(X_train_t, y_train - 1, sample_weight=sample_weights)

# Wrap predict so it returns original labels (1/2/3)
class XGBWrapper:
    def __init__(self, model):
        self.model = model
    def predict(self, X):
        return self.model.predict(X) + 1
    def predict_proba(self, X):
        return self.model.predict_proba(X)

xgb_wrapped = XGBWrapper(xgb)

xgb_metrics = evaluate('XGBoost', xgb_wrapped, X_train_t, y_train, X_test_t, y_test)
all_metrics.append(xgb_metrics)

## 4. Model comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Model':    m['model_name'],
        'Macro-F1': m['macro_f1'],
        'F1 (I)':   m['per_class']['1']['f1'],
        'F1 (II)':  m['per_class']['2']['f1'],
        'F1 (III)': m['per_class']['3']['f1'],
    }
    for m in all_metrics
])

print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
comparison.set_index('Model')[['Macro-F1', 'F1 (I)', 'F1 (II)', 'F1 (III)']].plot(
    kind='bar', ax=ax, rot=0
)
ax.set_ylim(0, 1)
ax.set_title('Model comparison — F1 scores (test set)')
ax.set_ylabel('F1')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUT_DIR / 'model_comparison.png', dpi=110)
plt.show()

## 5. Save best model → `backend/app/ml/model.pkl`

We pick the model with the highest Macro-F1 automatically.

In [ ]:
best = max(all_metrics, key=lambda m: m['macro_f1'])
print(f"Best model: {best['model_name']}  (Macro-F1 = {best['macro_f1']})")

model_map = {
    'Logistic Regression': lr,
    'Random Forest':       rf,
    'XGBoost':             xgb_wrapped,
}
best_model = model_map[best['model_name']]

joblib.dump(best_model, OUT_DIR / 'model.pkl')
joblib.dump(best_model, BACKEND_ML_DIR / 'model.pkl')
print('model.pkl saved to outputs/ and backend/app/ml/')

## 6. Save metrics → `outputs/metrics.json`

FastAPI's `/api/v1/metrics` endpoint reads this file (or the DB `model_versions` table).

In [ ]:
metrics_payload = {
    'best_model': best['model_name'],
    'all_models': all_metrics,
}

with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_payload, f, indent=2)

print(json.dumps(metrics_payload, indent=2))

## Summary

| Artifact | Location | Used by |
|----------|----------|---------|
| `model.pkl` | `backend/app/ml/` | FastAPI `prediction_service.py` |
| `pipeline.pkl` | `backend/app/ml/` | FastAPI `prediction_service.py` |
| `metrics.json` | `outputs/` | FastAPI `metrics_service.py` |
| `model_comparison.png` | `outputs/` | Presentation |
| `cm_*.png` | `outputs/` | Presentation |

**Inference flow** (matches `backend/app/services/prediction_service.py`):
```
raw input dict
  → pipeline.pkl.transform()
  → model.pkl.predict_proba()
  → { predicted_class, confidence, probabilities }
```
